# 08e_food_phytochemical_screen — 식품 지향 재스크리닝 (신규)

**한 줄 요약:** 대량생산 가능하고 안전한 **유명 식품 파이토케미컬**(플라보노이드·폴리페놀 등, 다수가 이미 한국 건기식 원료)에 신모델을 적용해 HSD17B13 저해 가능성을 순위 매긴다.
**왜:** 앞선 스크리닝 후보는 독성 미생물/해양 물질이라 건기식 부적합. 건기식은 **안전·식용·대량생산**이 우선 → 검증된 식품 성분에서 찾는다.
**주의:** 모델은 합성 active로 학습돼 식품 성분엔 점수가 낮게(보수적) 나올 수 있음. **효능 확신이 아니라 "상대 순위" 참고용.**
**큰 흐름:** ① 준비 → ② 모델·NP 로드 → ③ 후보 점수·순위

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기

In [ ]:
import os, sys
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로
print('작업 폴더:', os.getcwd())
import pickle, numpy as np, pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys, Descriptors, RDConfig
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
sys.path.append(os.path.join(RDConfig.RDContribDir,'NP_Score')); import npscorer

🔎 **코드 뜯어보기 (준비)**
- 신모델(08b·23) 예측기 + NP-likeness(08c) + RDKit 지문/descriptor. SMILES는 PubChem에서 받아 아래에 박아둠(재현성).

### 셀 1 — 모델 + 표현 계산기
표현앙상블과 4개 표현(maccs/rdkit/topotorsion/desc2d) 계산기, NP 점수 모델을 준비.

In [ ]:
# 신모델(표현앙상블) 로드 + 표현 계산기 + NP-likeness 모델
with open('data/HSD17B13_repr_ensemble.pkl','rb') as f: B=pickle.load(f)
TOP,REPS,PIPES=B['top'],B['reps'],B['pipelines']; NB=1024
D2=REPS['desc2d']; D2F=[getattr(Descriptors,n) for n in D2]   # 36개 개별 계산 함수
grd=rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NB)
gtt=rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=NB)
npm=npscorer.readNPModel()
def _b(fp,n):
    a=np.zeros((n,),dtype=np.int8); DataStructs.ConvertToNumpyArray(fp,a); return a
def rep(mols,r):
    if r=='maccs': return np.vstack([_b(MACCSkeys.GenMACCSKeys(m),167) for m in mols]).astype(np.float32)
    if r=='rdkit': return np.vstack([grd.GetFingerprintAsNumPy(m) for m in mols]).astype(np.float32)
    if r=='topotorsion': return np.vstack([gtt.GetFingerprintAsNumPy(m) for m in mols]).astype(np.float32)
    if r=='desc2d':
        a=np.array([[f(m) for f in D2F] for m in mols],dtype=np.float64); a[~np.isfinite(a)]=np.nan; return a
def predict(mols):
    need=sorted(set(r for _,r in TOP)); mats={r:rep(mols,r) for r in need}
    return np.mean([PIPES[f'{mn}|{rn}'].predict_proba(mats[rn])[:,1] for mn,rn in TOP],axis=0)

🔎 **코드 뜯어보기 (셀 1)**
- `predict(mols)` : 상위 조합들이 각자 표현으로 예측한 확률의 평균(소프트보팅). `rep(...)`은 08b와 동일 순서.

### 셀 2 — 후보 점수·순위
30종 식품 파이토케미컬을 모델로 예측하고, 천연물다움·분자량·할로겐(안전성)과 함께 순위.

In [ ]:
# 후보 파이토케미컬 예측: HSD17B13 활성확률 + 천연물다움 + 합성지표(안전성 참고)
COMPOUNDS = [
    ('Quercetin', '플라보놀·항산화(양파/사과)', 'C1=CC(=C(C=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O)O'),
    ('Kaempferol', '플라보놀(채소)', 'C1=CC(=CC=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O'),
    ('Myricetin', '플라보놀', 'C1=C(C=C(C(=C1O)O)O)C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O'),
    ('Isorhamnetin', '플라보놀', 'COC1=C(C=CC(=C1)C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O'),
    ('Luteolin', '플라본', 'C1=CC(=C(C=C1C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O)O'),
    ('Apigenin', '플라본(파슬리)', 'C1=CC(=CC=C1C2=CC(=O)C3=C(C=C(C=C3O2)O)O)O'),
    ('Baicalein', '플라본(황금)', 'C1=CC=C(C=C1)C2=CC(=O)C3=C(O2)C=C(C(=C3O)O)O'),
    ('Naringenin', '플라바논(감귤)', 'C1[C@H](OC2=CC(=CC(=C2C1=O)O)O)C3=CC=C(C=C3)O'),
    ('Hesperetin', '플라바논(감귤)', 'COC1=C(C=C(C=C1)[C@@H]2CC(=O)C3=C(C=C(C=C3O2)O)O)O'),
    ('Hesperidin', '감귤 배당체·대량생산 용이', 'C[C@H]1[C@@H]([C@H]([C@H]([C@@H](O1)OC[C@@H]2[C@H]([C@@H]([C@H]([C@@H](O2)OC3=CC(=C4C(=O)C[C@H](OC4=C3)C5=CC(=C(C=C5)OC)O)O)O)O)O)O)O)O'),
    ('Rutin', '루틴·건기식(메밀)', 'C[C@H]1[C@@H]([C@H]([C@H]([C@@H](O1)OC[C@@H]2[C@H]([C@@H]([C@H]([C@@H](O2)OC3=C(OC4=CC(=CC(=C4C3=O)O)O)C5=CC(=C(C=C5)O)O)O)O)O)O)O)O'),
    ('Naringin', '감귤 배당체', 'C[C@H]1[C@@H]([C@H]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@@H]([C@H](O[C@H]2OC3=CC(=C4C(=O)C[C@H](OC4=C3)C5=CC=C(C=C5)O)O)CO)O)O)O)O)O'),
    ('Catechin', '카테킨(녹차)', 'C1[C@@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C=C3)O)O)O'),
    ('Epicatechin', '카테킨(녹차/코코아)', 'C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C=C3)O)O)O'),
    ('Epigallocatechin gallate', 'EGCG·녹차 건기식 ★', 'C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C(=C3)O)O)O)OC(=O)C4=CC(=C(C(=C4)O)O)O'),
    ('Epigallocatechin', 'EGC(녹차)', 'C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C(=C3)O)O)O)O'),
    ('Genistein', '이소플라본(대두)', 'C1=CC(=CC=C1C2=COC3=CC(=CC(=C3C2=O)O)O)O'),
    ('Daidzein', '이소플라본(대두)', 'C1=CC(=CC=C1C2=COC3=C(C2=O)C=CC(=C3)O)O'),
    ('Resveratrol', '스틸벤·건기식(포도)', 'C1=CC(=CC=C1/C=C/C2=CC(=CC(=C2)O)O)O'),
    ('Pterostilbene', '스틸벤(블루베리)', 'COC1=CC(=CC(=C1)/C=C/C2=CC=C(C=C2)O)OC'),
    ('Curcumin', '강황·건기식 ★', 'COC1=C(C=CC(=C1)/C=C/C(=O)CC(=O)/C=C/C2=CC(=C(C=C2)O)OC)O'),
    ('Chlorogenic acid', '클로로겐산·체지방 건기식(커피)', 'C1[C@H]([C@H]([C@@H](C[C@@]1(C(=O)O)O)OC(=O)/C=C/C2=CC(=C(C=C2)O)O)O)O'),
    ('Caffeic acid', '페놀산', 'C1=CC(=C(C=C1/C=C/C(=O)O)O)O'),
    ('Ferulic acid', '페룰산(현미)', 'COC1=C(C=CC(=C1)/C=C/C(=O)O)O'),
    ('Gallic acid', '갈산', 'C1=C(C=C(C(=C1O)O)O)C(=O)O'),
    ('Ellagic acid', '엘라그산(석류)', 'C1=C2C3=C(C(=C1O)O)OC(=O)C4=CC(=C(C(=C43)OC2=O)O)O'),
    ('Silibinin', '실리비닌·밀크씨슬 간건강 건기식 ★★(MASLD 관련)', 'COC1=C(C=CC(=C1)[C@@H]2[C@H](OC3=C(O2)C=C(C=C3)[C@@H]4[C@H](C(=O)C5=C(C=C(C=C5O4)O)O)O)CO)O'),
    ('Sulforaphane', '설포라판(브로콜리)', 'CS(=O)CCCCN=C=S'),
    ('Hydroxytyrosol', '올리브 폴리페놀', 'C1=CC(=C(C=C1CCO)O)O'),
    ('Fisetin', '플라보놀(딸기)', 'C1=CC(=C(C=C1C2=C(C(=O)C3=C(O2)C=C(C=C3)O)O)O)O'),
]
halo=Chem.MolFromSmarts('[Cl,Br,I,F]')
rows=[]
for nm,note,smi in COMPOUNDS:
    m=Chem.MolFromSmiles(smi)
    if not m: continue
    rows.append({'이름':nm,'HSD17B13_prob':None,'NP':round(npscorer.scoreMol(m,npm),2),
                 'MW':round(Descriptors.MolWt(m),0),'할로겐':m.HasSubstructMatch(halo),
                 'SMILES':Chem.MolToSmiles(m),'비고':note,'_m':m})
mols=[r.pop('_m') for r in rows]
probs=predict(mols)
for r,p in zip(rows,probs): r['HSD17B13_prob']=round(float(p),3)
res=pd.DataFrame(rows).sort_values('HSD17B13_prob',ascending=False).reset_index(drop=True)
res.drop(columns=['SMILES']).to_csv('data/food_phytochemical_scores.csv',index=False)
print('식품 파이토케미컬 %d종 모델 순위 (HSD17B13 활성확률순):' % len(res))
print(res[['이름','HSD17B13_prob','NP','MW','할로겐','비고']].to_string(index=False))
print('\n저장: data/food_phytochemical_scores.csv')

🔎 **코드 뜯어보기 (셀 2)**
- `COMPOUNDS`=이름·메모·SMILES 목록(PubChem에서 확보). `predict(mols)`로 활성확률, `scoreMol`로 천연물다움. HSD17B13_prob 내림차순 정렬.